In [ ]:
 !pip install -q reportlab

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
import json
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

# ==========================================
# 1. QUESTION BANK
# ==========================================

quiz_data = {
    "Data Structures": [
        ("Which structure stores ordered values?",
         ["List", "Integer", "Boolean", "Float"], 0,
         "A list stores multiple values in an ordered collection."),

        ("Which collection is immutable?",
         ["List", "Tuple", "Set", "Dictionary"], 1,
         "A tuple cannot be modified after creation."),

        ("Which structure stores key-value pairs?",
         ["List", "Set", "Dictionary", "Tuple"], 2,
         "Dictionary stores data as key-value pairs."),

        ("Which collection removes duplicates?",
         ["List", "Tuple", "Set", "String"], 2,
         "A set stores unique elements."),

        ("Which symbol creates a list?",
         ["{}", "()", "[]", "<>"], 2,
         "Square brackets are used to create a list."),

        ("Which method adds an item to a list?",
         ["append()", "remove()", "clear()", "pop()"], 0,
         "append() adds an element at the end of a list.")
    ],

    "Loops": [
        ("Which loop iterates through a sequence?",
         ["if", "for", "try", "def"], 1,
         "A for loop iterates through sequence elements."),

        ("Which loop executes while a condition is true?",
         ["for", "while", "if", "class"], 1,
         "A while loop repeats while its condition is true."),

        ("Which statement terminates a loop?",
         ["continue", "pass", "break", "return"], 2,
         "break immediately terminates the current loop."),

        ("Which statement skips the current iteration?",
         ["break", "continue", "pass", "exit"], 1,
         "continue skips the remaining statements in the current iteration."),

        ("What does range(5) produce?",
         ["1 to 5", "0 to 4", "0 to 5", "1 to 4"], 1,
         "range(5) generates integers from 0 through 4."),

        ("Which keyword represents an empty statement?",
         ["skip", "empty", "pass", "null"], 2,
         "pass is a placeholder that performs no operation.")
    ],

    "Functions": [
        ("Which keyword defines a function?",
         ["function", "def", "fun", "define"], 1,
         "Python uses def to define a function."),

        ("Which keyword returns a value?",
         ["print", "input", "return", "break"], 2,
         "return sends a result back to the caller."),

        ("What is a parameter?",
         ["Variable in function definition",
          "Loop", "Class", "Module"], 0,
         "A parameter receives a value when a function is called."),

        ("Which accepts multiple positional arguments?",
         ["**kwargs", "*args", "lambda", "return"], 1,
         "*args collects extra positional arguments into a tuple."),

        ("Which creates an anonymous function?",
         ["def", "class", "lambda", "pass"], 2,
         "lambda creates a small anonymous function."),

        ("A function calling itself is called?",
         ["Iteration", "Recursion", "Inheritance", "Overloading"], 1,
         "Recursion occurs when a function calls itself.")
    ],

    "OOP": [
        ("What does OOP stand for?",
         ["Object Oriented Programming",
          "Open Object Program",
          "Order of Operations",
          "Object Output Programming"], 0,
         "OOP means Object-Oriented Programming."),

        ("Which keyword creates a class?",
         ["object", "class", "def", "self"], 1,
         "The class keyword defines a class."),

        ("Which method initializes an object?",
         ["start()", "main()", "__init__()", "create()"], 2,
         "__init__ is the constructor method."),

        ("What is inheritance?",
         ["Deleting objects",
          "Reusing class properties and methods",
          "Creating loops",
          "Opening files"], 1,
         "Inheritance allows a child class to reuse parent class features."),

        ("What does self represent?",
         ["Parent class", "Current instance", "Module", "Function"], 1,
         "self refers to the current object instance."),

        ("Which concept hides internal implementation?",
         ["Inheritance", "Polymorphism",
          "Encapsulation", "Iteration"], 2,
         "Encapsulation combines data and methods and controls access.")
    ],

    "File Handling": [
        ("Which function opens a file?",
         ["read()", "open()", "file()", "write()"], 1,
         "open() is used to open a file."),

        ("Which mode reads a file?",
         ["w", "a", "r", "x"], 2,
         "r is the read mode."),

        ("Which mode overwrites existing content?",
         ["r", "w", "a", "x"], 1,
         "w creates or truncates a file for writing."),

        ("Which method reads the entire file?",
         ["read()", "write()", "close()", "append()"], 0,
         "read() returns the file content."),

        ("Which method closes a file?",
         ["stop()", "end()", "close()", "exit()"], 2,
         "close() closes an opened file."),

        ("Which keyword manages file closing automatically?",
         ["with", "try", "finally", "open"], 0,
         "with uses a context manager to close the file automatically.")
    ],

    "Exception Handling": [
        ("Which block handles exceptions?",
         ["try", "except", "raise", "finally"], 1,
         "except handles exceptions raised in the try block."),

        ("Which block contains risky code?",
         ["except", "finally", "try", "else"], 2,
         "The try block contains code that might raise an exception."),

        ("Which keyword manually raises an exception?",
         ["throw", "raise", "error", "break"], 1,
         "raise is used to trigger an exception."),

        ("Which block normally runs whether an exception occurs or not?",
         ["try", "except", "else", "finally"], 3,
         "finally executes during normal and exceptional control flow."),

        ("Which exception occurs when dividing by zero?",
         ["ValueError", "TypeError",
          "ZeroDivisionError", "NameError"], 2,
         "ZeroDivisionError occurs during division by zero."),

        ("How do you define a custom exception?",
         ["Create a class derived from Exception",
          "Use a for loop",
          "Use print()",
          "Use input()"], 0,
         "A custom exception is normally defined using a class derived from Exception.")
    ]
}

# ==========================================
# 2. FILE STORAGE
# ==========================================

LEADERBOARD_FILE = "leaderboard.csv"

def load_scores():
    if os.path.exists(LEADERBOARD_FILE):
        return pd.read_csv(LEADERBOARD_FILE)
    return pd.DataFrame(
        columns=["Name", "Topic", "Score", "Percentage", "Date"]
    )


def save_score(name, topic, score, percentage):
    df = load_scores()

    new_record = pd.DataFrame([{
        "Name": name,
        "Topic": topic,
        "Score": score,
        "Percentage": round(percentage, 2),
        "Date": datetime.now().strftime("%Y-%m-%d %H:%M")
    }])

    df = pd.concat([df, new_record], ignore_index=True)
    df.to_csv(LEADERBOARD_FILE, index=False)


# ==========================================
# 3. AI QUIZ GENERATOR
# ==========================================

def generate_ai_questions(topic, count=10):
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=api_key)

            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Create Python programming MCQs. "
                            "Return only a valid JSON array. "
                            "Each item must have question, options "
                            "(four strings), correct_index "
                            "(0 to 3), and explanation."
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Generate {count} beginner-to-intermediate "
                            f"questions about Python {topic}. "
                            "Do not repeat basic questions."
                        )
                    }
                ],
                response_format={"type": "json_object"}
            )

            data = json.loads(response.choices[0].message.content)
            questions = data["questions"]

            if len(questions) != count:
                raise ValueError("Unexpected question count")

            for q in questions:
                if (
                    len(q["options"]) != 4
                    or not 0 <= q["correct_index"] < 4
                ):
                    raise ValueError("Invalid question format")

            return [
                (
                    q["question"],
                    q["options"],
                    q["correct_index"],
                    q["explanation"]
                )
                for q in questions
            ]

        except Exception as error:
            print("AI generation unavailable:", error)

    # Fallback pool from all topics if selected topic is All
    if topic == "All Topics":
        all_pool = []
        for t in quiz_data.values():
            all_pool.extend(t)
        return random.sample(all_pool, min(count, len(all_pool)))
    return random.sample(quiz_data[topic], min(count, len(quiz_data[topic])))


# ==========================================
# 4. GENERATE QUIZ
# ==========================================

def start_quiz(topic, mode):
    # If topic is All Topics, we pull questions across all categories
    if topic == "All Topics":
        all_questions = []
        for t_questions in quiz_data.values():
            all_questions.extend(t_questions)
        if mode == "AI Generated":
            questions = generate_ai_questions("All Topics", count=10)
        else:
            questions = random.sample(all_questions, 10)
    else:
        if mode == "AI Generated":
            questions = generate_ai_questions(topic, count=10)
        else:
            questions = random.sample(quiz_data[topic], 6)

    return questions, 300, "Quiz started! You have 5 minutes."


# ==========================================
# 5. EXAM TIMER
# ==========================================

def countdown(seconds):
    if seconds <= 0:
        return 0, "Time is up! Submit your answers."

    seconds -= 1

    minutes = seconds // 60
    remaining_seconds = seconds % 60

    message = (
        f"Time Remaining: {minutes:02d}:"
        f"{remaining_seconds:02d}"
    )

    return seconds, message


# ==========================================
# 6. PDF CERTIFICATE
# ==========================================

def create_certificate(name, score, percentage, topic):
    filename = "certificate_" + str(datetime.now().timestamp()) + ".pdf"

    pdf = canvas.Canvas(filename, pagesize=A4)
    width, height = A4

    pdf.setTitle("Python Quiz Certificate")

    pdf.setStrokeColorRGB(0.1, 0.3, 0.6)
    pdf.setLineWidth(5)
    pdf.rect(30, 30, width - 60, height - 60)

    pdf.setFont("Helvetica-Bold", 26)
    pdf.drawCentredString(
        width / 2, height - 150,
        "CERTIFICATE OF ACHIEVEMENT"
    )

    pdf.setFont("Helvetica", 16)
    pdf.drawCentredString(
        width / 2, height - 220,
        "This certificate is proudly presented to"
    )

    pdf.setFont("Helvetica-Bold", 25)
    pdf.drawCentredString(
        width / 2, height - 275, name
    )

    pdf.setFont("Helvetica", 15)
    pdf.drawCentredString(
        width / 2, height - 340,
        "For completing the Python MCQ Assessment"
    )

    pdf.drawCentredString(
        width / 2, height - 390,
        f"Topic: {topic}"
    )

    pdf.drawCentredString(
        width / 2, height - 430,
        f"Final Grade: {percentage:.2f}/100"
    )

    pdf.drawCentredString(
        width / 2, height - 500,
        datetime.now().strftime("%d-%m-%Y")
    )

    pdf.setFont("Helvetica-Oblique", 12)
    pdf.drawCentredString(
        width / 2, 100,
        "Python Smart Learning Assessment System"
    )

    pdf.save()

    return filename


# ==========================================
# 7. PERFORMANCE ANALYTICS
# ==========================================

def create_analytics():
    df = load_scores()

    if df.empty:
        return None, "No student records available yet."

    topic_average = df.groupby("Topic")["Percentage"].mean()

    fig, ax = plt.subplots(figsize=(9, 4))
    topic_average.plot(kind="bar", ax=ax)

    ax.set_title("Average Performance by Topic")
    ax.set_ylabel("Percentage (Scaled to 100)")
    ax.set_ylim(0, 100)
    ax.set_xlabel("Python Topics")

    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

    summary = f"""
    ### Performance Analytics

    - Total Attempts: {len(df)}
    - Average Percentage: {df['Percentage'].mean():.2f}/100
    - Highest Score: {df['Score'].max()}/100
    - Weakest Topic: {topic_average.idxmin()}
    - Strongest Topic: {topic_average.idxmax()}
    """

    return fig, summary


# ==========================================
# 8. AI LEARNING ASSISTANT
# ==========================================

def learning_assistant(question, explanation, correct_answer):
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=api_key)

            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a friendly Python programming tutor. "
                            "Explain concepts simply with a beginner-level "
                            "example and real-world application."
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Question: {question}\n"
                            f"Correct answer: {correct_answer}\n"
                            f"Explanation: {explanation}"
                        )
                    }
                ]
            )

            return response.choices[0].message.content

        except Exception as error:
            print("AI assistant unavailable:", error)

    return f"""
    ### Python Learning Assistant

    **Question:** {question}

    **Correct Answer:** {correct_answer}

    **Explanation:** {explanation}

    **Learning Tip:** Practise this concept using a small Python
    program and check its output.
    """


# ==========================================
# 9. SUBMIT QUIZ AND CALCULATE MARKS
# ==========================================

def submit_quiz(name, email, student_id, topic, questions, seconds, *answers):
    if not name.strip() or not email.strip() or not student_id.strip():
        return "Please enter all student details.", None, None, None

    if not questions:
        return "Please generate a quiz first.", None, None, None

    if seconds <= 0:
        return "Time is over. Your answers cannot be submitted.", None, None, None

    raw_score = 0
    review = []

    for i, q in enumerate(questions):
        question, options, correct_index, explanation = q
        user_answer = answers[i]

        if user_answer is not None and user_answer == options[correct_index]:
            raw_score += 1
            status = "Correct"
        else:
            status = "Incorrect"

        review.append(
            f"""
            **Q{i+1}. {question}**

            Your Answer: {user_answer or 'Not Answered'}

            Correct Answer: {options[correct_index]}

            Status: {status}

            Explanation: {explanation}

            ---
            """
        )

    total_questions = len(questions)
    # Convert percentage directly to score out of 100 grade scale
    percentage = (raw_score / total_questions) * 100
    score_out_of_100 = round(percentage, 2)

    save_score(name, topic, score_out_of_100, percentage)

    performance = (
        "Excellent" if percentage >= 80 else
        "Good" if percentage >= 60 else
        "Average" if percentage >= 40 else
        "Needs Improvement"
    )

    result = f"""
    # STUDENT MARK STATEMENT

    **Name:** {name}

    **Email:** {email}

    **Student ID:** {student_id}

    **Topic:** {topic}

    | Assessment | Result |
    |---|---:|
    | Total Questions | {total_questions} |
    | Correct Answers | {raw_score} |
    | Wrong Answers | {total_questions - raw_score} |
    | Final Grade | {score_out_of_100}/100 |
    | Percentage | {percentage:.2f}% |

    ## Performance: {performance}

    ## Answer Review

    """ + "\n".join(review)

    certificate = None
    if percentage >= 60:
        certificate = create_certificate(
            name, score_out_of_100, percentage, topic
        )

    return result, certificate, *create_analytics()


# ==========================================
# 10. GRADIO APPLICATION
# ==========================================

with gr.Blocks(
    title="Python Smart Learning"
) as app:

    gr.Markdown("""
    # 🐍 PYTHON SMART LEARNING & ASSESSMENT SYSTEM

    ### Learn Python | Take Quizzes | Analyse Performance | Earn Certificates
    """)

    questions_state = gr.State([])
    remaining_state = gr.State(300)

    with gr.Tab("Student Registration & Quiz"):

        gr.Markdown("## 1. Student Details")

        name = gr.Textbox(label="Student Name")
        email = gr.Textbox(label="Email Address")
        student_id = gr.Textbox(label="Student ID")

        gr.Markdown("## 2. Select Quiz")

        # Added 'All Topics' option to allow taking quiz on everything
        topic = gr.Dropdown(
            choices=list(quiz_data.keys()) + ["All Topics"],
            value="All Topics",
            label="Select Python Topic"
        )

        mode = gr.Radio(
            choices=["Question Bank", "AI Generated"],
            value="Question Bank",
            label="Question Generation Mode"
        )

        start_button = gr.Button(
            "Generate Quiz",
            variant="primary"
        )

        status = gr.Markdown()
        timer_display = gr.Markdown("Timer: 05:00")
        timer = gr.Timer(1, active=False)

        @gr.render(inputs=questions_state)
        def display_questions(questions):
            if questions:
                gr.Markdown("## 3. Answer Your Questions")
                answer_components = []

                for i, q in enumerate(questions):
                    question, options, correct, explanation = q
                    answer = gr.Radio(
                        choices=options,
                        label=f"Q{i+1}. {question}"
                    )
                    answer_components.append(answer)

                submit_button = gr.Button(
                     "Submit Quiz",
                     variant="primary"
                )

                result_output = gr.Markdown()
                certificate_output = gr.File(
                    label="Download Certificate"
                )
                analytics_plot = gr.Plot(
                    label="Performance Chart"
                )
                analytics_text = gr.Markdown()

                submit_button.click(
                    fn=submit_quiz,
                    inputs=[
                        name, email, student_id,
                        topic, questions_state, remaining_state
                    ] + answer_components,
                    outputs=[
                        result_output,
                        certificate_output,
                        analytics_plot,
                        analytics_text
                    ]
                )

        def begin(topic, mode):
            questions, seconds, message = start_quiz(topic, mode)
            return questions, seconds, message

        start_button.click(
            fn=begin,
            inputs=[topic, mode],
            outputs=[questions_state, remaining_state, status]
        ).then(
            fn=lambda: gr.Timer(active=True),
            outputs=timer
        )

        def update_timer(seconds):
            new_seconds, message = countdown(seconds)
            return new_seconds, message

        timer.tick(
            fn=update_timer,
            inputs=remaining_state,
            outputs=[remaining_state, timer_display]
        )

    with gr.Tab("Leaderboard"):
        gr.Markdown("## 🏆 Student Leaderboard")
        leaderboard_output = gr.Dataframe(
            value=load_scores().sort_values(
                "Percentage", ascending=False
            ),
            label="Student Rankings",
            interactive=False
        )
        refresh_button = gr.Button("Refresh Leaderboard")
        refresh_button.click(
            fn=lambda: load_scores().sort_values(
                "Percentage", ascending=False
            ),
            outputs=leaderboard_output
        )

    with gr.Tab("Performance Analytics"):
        gr.Markdown("## 📊 Overall Performance")
        analytics_button = gr.Button("Generate Analytics")
        chart_output = gr.Plot()
        analytics_output = gr.Markdown()

        analytics_button.click(
            fn=create_analytics,
            outputs=[chart_output, analytics_output]
        )

    with gr.Tab("AI Learning Assistant"):
        gr.Markdown("""
        ## 🤖 Python Learning Assistant

        Ask the assistant to explain a Python concept.
        """)

        assistant_question = gr.Textbox(
            label="Question",
            placeholder="Example: What is inheritance?"
        )
        assistant_answer = gr.Textbox(
            label="Correct Answer / Concept",
            placeholder="Example: Inheritance"
        )
        assistant_explanation = gr.Textbox(
            label="Additional Explanation (optional)"
        )

        assistant_button = gr.Button("Explain Concept")
        assistant_output = gr.Markdown()

        assistant_button.click(
            fn=learning_assistant,
            inputs=[
                assistant_question,
                assistant_explanation,
                assistant_answer
            ],
            outputs=assistant_output
        )

# ==========================================
# 11. RUN APPLICATION
# ==========================================

app.launch(theme=gr.themes.Soft())
